# WESAD preprocessing and leakage checks

This notebook uses the corrected compact CSV windows prepared from the WESAD wrist sensors. It verifies schema, labels, missing/non-finite values, duplicate rows, ordering, and participant separation, then runs leave-one-participant-out (LOSO) evaluation.

**Important:** the supplied data contains only 15 people and recordings from a controlled lab protocol. The LOSO results measure performance on an unseen participant from this dataset; they do not establish real-world or live interface performance.

In [ ]:
%pip -q install pandas numpy scikit-learn

## 1. Load the corrected CSVs from Google Drive

Set this to the Drive folder containing `S2_windows.csv`, `S3_windows.csv`, etc. The files must come from the corrected preprocessing script, where label windows use the 700 Hz synchronized label stream.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import pandas as pd
import numpy as np

DATA_DIR = Path('/content/drive/MyDrive/DataCon/Project 6 Wesad')
files = sorted(DATA_DIR.glob('*_windows.csv'))
if not files:
    raise FileNotFoundError(f'No *_windows.csv files in {DATA_DIR}')
df = pd.concat((pd.read_csv(p) for p in files), ignore_index=True)
print('Files:', len(files), '| shape:', df.shape)
display(df.head())

## 2. Schema and data-quality audit

The raw WESAD wrist channels have different sample rates: ACC 32 Hz, BVP 64 Hz, EDA and TEMP 4 Hz. The synchronized labels are 700 Hz. Correct windowing must use the label rate for labels and each sensor's own rate for sensor summaries.

In [ ]:
required = {'subject','window','label','label_id'}
assert required.issubset(df.columns), f'Missing columns: {required - set(df.columns)}'
feature_cols = [c for c in df.columns if c not in required]
assert feature_cols, 'No feature columns found'
print('Participants:', df.subject.nunique(), sorted(df.subject.unique()))
print('Label names:', sorted(df.label.dropna().unique()))
print('Label IDs:', sorted(df.label_id.dropna().unique()))
print('Missing feature values:', int(df[feature_cols].isna().sum().sum()))
finite = np.isfinite(df[feature_cols].to_numpy(dtype=float))
print('Non-finite feature values:', int((~finite).sum()))
print('Duplicate full rows:', int(df.duplicated().sum()))
print('Label to ID mapping:')
display(df.groupby('label').label_id.unique())
print('Windows per subject:')
display(df.groupby('subject').size().rename('windows').to_frame())
assert df[feature_cols].notna().all().all(), 'Missing features found; investigate before modeling'
assert finite.all(), 'Infinite feature values found; investigate before modeling'
assert df.groupby('label').label_id.nunique().max() == 1, 'A label maps to multiple IDs'
assert df.duplicated().sum() == 0, 'Duplicate rows found'

## 3. Verify time order and window construction

Rows should remain chronological within each participant. The window index is recorded in ten-second steps; gaps can occur because windows dominated by transition labels were intentionally excluded.

In [ ]:
order_ok = df.groupby('subject')['window'].apply(lambda s: s.is_monotonic_increasing)
print('All participant files ordered:', bool(order_ok.all()))
assert order_ok.all(), 'Rows are not chronological within one or more participants'
print('Label counts:')
display(df.label.value_counts().rename('windows').to_frame())
print('Class proportions:')
display(df.label.value_counts(normalize=True).round(3).rename('proportion').to_frame())
print('Labels by participant:')
display(pd.crosstab(df.subject, df.label))

## 4. Prevent subject leakage with LOSO evaluation

A random row split can put windows from the same participant in both train and test. Because physiological baselines and sensor placement differ by person, that can inflate performance. This loop holds out one entire participant at a time. Imputation and scaling are fitted on the training participants only inside each fold.

The model here is a simple Logistic Regression baseline. Replace it with an online learner for subsequent streaming experiments, while keeping the same participant-held-out evaluation principle.

In [ ]:
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

X = df[feature_cols].astype(float)
y = df['label'].astype(str)
groups = df['subject'].astype(str)
logo = LeaveOneGroupOut()
rows, all_true, all_pred = [], [], []
for train_idx, test_idx in logo.split(X, y, groups):
    train_subjects = set(groups.iloc[train_idx])
    test_subjects = set(groups.iloc[test_idx])
    assert train_subjects.isdisjoint(test_subjects), 'Subject leakage in fold'
    model = make_pipeline(
        SimpleImputer(strategy='median'),
        StandardScaler(),
        LogisticRegression(max_iter=2000, class_weight='balanced')
    )
    model.fit(X.iloc[train_idx], y.iloc[train_idx])
    pred = model.predict(X.iloc[test_idx])
    truth = y.iloc[test_idx].to_numpy()
    rows.append({
        'held_out_subject': next(iter(test_subjects)),
        'n_test_windows': len(test_idx),
        'accuracy': accuracy_score(truth, pred),
        'macro_f1': f1_score(truth, pred, average='macro', zero_division=0),
    })
    all_true.extend(truth)
    all_pred.extend(pred)

results = pd.DataFrame(rows).sort_values('held_out_subject')
display(results)
print('Mean LOSO accuracy:', round(results.accuracy.mean(), 3), '+/-', round(results.accuracy.std(), 3))
print('Mean LOSO macro F1:', round(results.macro_f1.mean(), 3), '+/-', round(results.macro_f1.std(), 3))
print('Pooled out-of-subject report:')
print(classification_report(all_true, all_pred, labels=['baseline','stress','amusement'], zero_division=0))
print('Pooled confusion matrix (rows=true, columns=predicted):')
display(pd.DataFrame(confusion_matrix(all_true, all_pred, labels=['baseline','stress','amusement']), index=['true_baseline','true_stress','true_amusement'], columns=['pred_baseline','pred_stress','pred_amusement']))

## 5. Leakage checklist and interpretation

- Do not randomly split rows from these repeated time windows. Keep each participant wholly inside one fold.
- Fit imputation, scaling, feature selection, and any resampling using training folds only. The pipeline above fits imputation and scaling separately per fold.
- Do not use `subject`, `window`, or `label_id` as predictors. They are identifiers, time order, and a duplicate encoding of the target.
- The class labels come from the experimental protocol and can occur in long blocks. This is acceptable for supervised benchmark classification, but it is not a stream of independent natural user feedback.
- The 10-second summaries are suitable for an initial model. A deployment claim requires a separate real-time implementation and usability study.

### Save fold results


In [ ]:
OUT = Path('/content/drive/MyDrive/DataCon/Project 6 Wesad')
results.to_csv(OUT / 'wesad_loso_results.csv', index=False)
print('Saved fold results to', OUT / 'wesad_loso_results.csv')